# VeriSight: FaceForensics++ Clean Deepfake Detection Benchmark
### Video-Disjoint Splitting, Frame Deduplication, and Two-Stream Spatial-Spectral Architecture
---
### Why this Notebook Resolves FaceForensics++ Limitations:
1. **Video-Disjoint Splitting**: Frames from the same source video appear in **only one** partition (Train, Val, or Test). This prevents the network from cheating by memorizing backgrounds or speaker identities.
2. **Frame Deduplication / Subsampling**: Removes consecutive near-duplicate video frames by sampling evenly spaced keyframes per video sequence.
3. **Balanced Multi-Technique Evaluation**: Evaluates authentic YouTube faces against all major manipulation algorithms (Deepfakes, Face2Face, FaceSwap, NeuralTextures, FaceShifter).
4. **State-of-the-Art Architecture**: Combines Spatial ResNet-50 with our Dedicated 4-Stage 2D FFT Spectral CNN.
5. **Zero-Leakage Evaluation**: Fusion weights are tuned strictly on Validation data and evaluated once on untouched Test data.

In [ ]:
# Step 1: Environment & GPU Verification
import torch
import numpy as np
import random
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU Architecture: {torch.cuda.get_device_name(0)}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/verisight_models')
except Exception:
    SAVE_DIR = Path('./saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Models will be exported to: {SAVE_DIR}')

In [ ]:
# Step 2: Download FaceForensics++ Extracted Benchmark via KaggleHub
!pip install -q kagglehub
import kagglehub

print('Downloading official FaceForensics++ C23 extracted frames benchmark...')
path_ff = kagglehub.dataset_download('fatimahirshad/faceforensics-extracted-dataset-c23')
print(f'FaceForensics++ downloaded successfully to: {path_ff}')

In [ ]:
# Step 3: Frame Deduplication and Video-Disjoint Partitioning
import os
from collections import defaultdict

root_ff = Path(path_ff)
all_files = [p for p in root_ff.rglob('*.*') if p.suffix.lower() in ['.jpg', '.png', '.jpeg']]
print(f'Total raw extracted frames discovered: {len(all_files):,}')

# Categorize by manipulation family
technique_map = {
    'Real_Original': [],
    'Deepfakes': [],
    'Face2Face': [],
    'FaceSwap': [],
    'NeuralTextures': [],
    'FaceShifter': []
}

for p in all_files:
    posix = p.as_posix().lower()
    if 'original' in posix or 'real' in posix:
        technique_map['Real_Original'].append(p)
    elif 'deepfakes' in posix:
        technique_map['Deepfakes'].append(p)
    elif 'face2face' in posix:
        technique_map['Face2Face'].append(p)
    elif 'faceswap' in posix:
        technique_map['FaceSwap'].append(p)
    elif 'neuraltextures' in posix:
        technique_map['NeuralTextures'].append(p)
    elif 'faceshifter' in posix:
        technique_map['FaceShifter'].append(p)

for t_name, files in technique_map.items():
    print(f'  - {t_name:<16}: {len(files):,} raw frames')

# FRAME DEDUPLICATION & VIDEO GROUPING:
# Group frames by base video identifier to ensure temporal deduplication and video-disjoint splits
def extract_video_id(path):
    # File pattern usually contains sequence IDs (e.g. 001_002 or video_012)
    stem = path.stem
    parts = stem.split('_')
    if len(parts) >= 2:
        return parts[0] + '_' + parts[1]
    return path.parent.name

def deduplicate_frames_per_video(file_list, max_frames_per_video=8):
    video_dict = defaultdict(list)
    for f in file_list:
        v_id = extract_video_id(f)
        video_dict[v_id].append(f)
    
    deduped = []
    for v_id, frames in video_dict.items():
        frames.sort() # chronological order
        # Subsample evenly to remove near-identical adjacent frames
        step = max(1, len(frames) // max_frames_per_video)
        selected = frames[::step][:max_frames_per_video]
        for s in selected:
            deduped.append((s, v_id))
    return deduped

real_dedup = deduplicate_frames_per_video(technique_map['Real_Original'], max_frames_per_video=8)
fake_dedup = []
for cat in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
    fake_dedup.extend(deduplicate_frames_per_video(technique_map[cat], max_frames_per_video=4))

print(f'\nAfter Temporal Deduplication:')
print(f'  Authentic Real Unique Keyframes: {len(real_dedup):,}')
print(f'  Manipulated Deepfake Keyframes:   {len(fake_dedup):,}')

# VIDEO-DISJOINT SPLIT (80% Train, 10% Val, 10% Test by Video ID):
all_real_vids = list(set([v_id for _, v_id in real_dedup]))
all_fake_vids = list(set([v_id for _, v_id in fake_dedup]))
random.shuffle(all_real_vids)
random.shuffle(all_fake_vids)

def split_vids(vids):
    n = len(vids)
    return set(vids[:int(0.80*n)]), set(vids[int(0.80*n):int(0.90*n)]), set(vids[int(0.90*n):])

train_rv, val_rv, test_rv = split_vids(all_real_vids)
train_fv, val_fv, test_fv = split_vids(all_fake_vids)

train_items = [item for item in real_dedup if item[1] in train_rv] + [item for item in fake_dedup if item[1] in train_fv]
val_items   = [item for item in real_dedup if item[1] in val_rv]   + [item for item in fake_dedup if item[1] in val_fv]
test_items  = [item for item in real_dedup if item[1] in test_rv]  + [item for item in fake_dedup if item[1] in test_fv]

random.shuffle(train_items)
random.shuffle(val_items)
random.shuffle(test_items)

def package_split(items):
    paths = [it[0] for it in items]
    labels = [1 if 'original' in it[0].as_posix().lower() or 'real' in it[0].as_posix().lower() else 0 for it in items]
    v_ids = [it[1] for it in items]
    return paths, labels, v_ids

train_p, train_y, _ = package_split(train_items)
val_p, val_y, _     = package_split(val_items)
test_p, test_y, _   = package_split(test_items)

print(f'\nStrict Video-Disjoint Splits Created:')
print(f'  Train Set: {len(train_p):,} frames (Real: {train_y.count(1):,}, Fake: {train_y.count(0):,})')
print(f'  Val Set:   {len(val_p):,} frames (Real: {val_y.count(1):,}, Fake: {val_y.count(0):,})')
print(f'  Test Set:  {len(test_p):,} frames (Real: {test_y.count(1):,}, Fake: {test_y.count(0):,})')

In [ ]:
# Step 4: Dataset Pipeline with Compression Equalization and Augmentation
import io
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def apply_compression_equalization(pil_img, quality=85):
    """Passes frames through uniform JPEG compression to eliminate video-codec shortcuts."""
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

class FaceForensicsDataset(Dataset):
    def __init__(self, paths, labels, transform=None, equalize_compression=True):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.equalize_compression = equalize_compression
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.equalize_compression:
            img = apply_compression_equalization(img, quality=85)
            
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx], str(path)

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

BATCH_SIZE = 64
train_loader = DataLoader(FaceForensicsDataset(train_p, train_y, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(FaceForensicsDataset(val_p, val_y, val_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(FaceForensicsDataset(test_p, test_y, val_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders compiled: {len(train_loader)} training batches.')

In [ ]:
# Step 5: Model Architectures (Spatial ResNet-50 + Dedicated 2D FFT Spectral CNN)
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class SpatialStreamResNet50(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = resnet.fc.in_features
        resnet.fc = nn.Identity()
        self.encoder = resnet
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

class DedicatedSpectralCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def extract_fft(self, x_rgb):
        inv_mean = torch.tensor([0.485, 0.456, 0.406], device=x_rgb.device).view(1, 3, 1, 1)
        inv_std = torch.tensor([0.229, 0.224, 0.225], device=x_rgb.device).view(1, 3, 1, 1)
        rgb = torch.clamp(x_rgb * inv_std + inv_mean, 0.0, 1.0)
        weights = torch.tensor([0.2989, 0.5870, 0.1140], device=rgb.device).view(1, 3, 1, 1)
        gray = torch.sum(rgb * weights, dim=1, keepdim=True)
        fft2 = torch.fft.fft2(gray)
        fft_shifted = torch.fft.fftshift(fft2, dim=(-2, -1))
        mag = torch.log1p(torch.abs(fft_shifted))
        b = mag.shape[0]
        min_v = mag.view(b, -1).min(dim=1)[0].view(b, 1, 1, 1)
        max_v = mag.view(b, -1).max(dim=1)[0].view(b, 1, 1, 1)
        return (mag - min_v) / (max_v - min_v + 1e-8)

    def forward(self, x_rgb):
        x_fft = self.extract_fft(x_rgb)
        return self.classifier(self.features(x_fft))

print('Spatial ResNet-50 and Dedicated 2D FFT CNN compiled.')

In [ ]:
# Step 6: Train Both Streams on FaceForensics++ (5 Epochs Each)
from sklearn.metrics import accuracy_score

model_spatial = SpatialStreamResNet50().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer_spat = torch.optim.AdamW(model_spatial.parameters(), lr=1e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda')

print('=== Training Spatial Stream on FaceForensics++ (5 Epochs) ===')
for epoch in range(1, 6):
    model_spatial.train()
    running_loss = 0.0
    for imgs, lbls, _ in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer_spat.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model_spatial(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer_spat)
        scaler.update()
        running_loss += loss.item()
        
    model_spatial.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, lbls, _ in val_loader:
            imgs = imgs.to(DEVICE)
            preds.extend(model_spatial(imgs).argmax(1).cpu().numpy())
            targets.extend(lbls.numpy())
    val_acc = accuracy_score(targets, preds)
    print(f'Epoch {epoch}/5: Train Loss={running_loss/len(train_loader):.4f} | Val Accuracy={val_acc*100:.2f}%')

model_spectral = DedicatedSpectralCNN().to(DEVICE)
optimizer_spec = torch.optim.AdamW(model_spectral.parameters(), lr=1e-3, weight_decay=1e-4)

print('\n=== Training Spectral Stream on FaceForensics++ (5 Epochs) ===')
for epoch in range(1, 6):
    model_spectral.train()
    running_loss = 0.0
    for imgs, lbls, _ in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer_spec.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model_spectral(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer_spec)
        scaler.update()
        running_loss += loss.item()
        
    model_spectral.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, lbls, _ in val_loader:
            imgs = imgs.to(DEVICE)
            preds.extend(model_spectral(imgs).argmax(1).cpu().numpy())
            targets.extend(lbls.numpy())
    val_acc = accuracy_score(targets, preds)
    print(f'Epoch {epoch}/5: Train Loss={running_loss/len(train_loader):.4f} | Val Accuracy={val_acc*100:.2f}%')

In [ ]:
# Step 7: Methodologically Sound Zero-Leakage Evaluation on FaceForensics++
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

model_spatial.eval()
model_spectral.eval()

# Phase 1: Tune Fusion Weight on VALIDATION Data
print('=== Phase 1: Tuning Fusion Weight on Validation Data ===')
val_spat, val_spec, val_true = [], [], []
with torch.no_grad():
    for imgs, lbls, _ in val_loader:
        imgs = imgs.to(DEVICE)
        val_spat.extend(F.softmax(model_spatial(imgs), dim=1)[:, 0].cpu().numpy())
        val_spec.extend(F.softmax(model_spectral(imgs), dim=1)[:, 0].cpu().numpy())
        val_true.extend(lbls.numpy())

val_spat = np.array(val_spat)
val_spec = np.array(val_spec)
val_true = np.array(val_true)

best_w, best_val_acc = 0.5, 0.0
for w in [0.95, 0.90, 0.85, 0.80, 0.70, 0.50]:
    val_p = w * val_spat + (1.0 - w) * val_spec
    val_preds = (val_p < 0.50).astype(int)
    acc_w = accuracy_score(val_true, val_preds)
    if acc_w > best_val_acc:
        best_val_acc = acc_w
        best_w = w

print(f'Optimal Weight Selected on Validation Set: {best_w*100:.0f}% Spatial + {(1.0-best_w)*100:.0f}% Spectral')
print(f'Validation Fused Accuracy: {best_val_acc*100:.2f}%')

# Phase 2: FREEZE best_w and Evaluate ONCE on Untouched Video-Disjoint Test Set
print('\n=== Phase 2: Independent Evaluation on Untouched Video-Disjoint Test Set ===')
test_spat, test_spec, test_true, test_paths = [], [], [], []
with torch.no_grad():
    for imgs, lbls, paths in test_loader:
        imgs = imgs.to(DEVICE)
        test_spat.extend(F.softmax(model_spatial(imgs), dim=1)[:, 0].cpu().numpy())
        test_spec.extend(F.softmax(model_spectral(imgs), dim=1)[:, 0].cpu().numpy())
        test_true.extend(lbls.numpy())
        test_paths.extend(paths)

test_spat = np.array(test_spat)
test_spec = np.array(test_spec)
test_true = np.array(test_true)

fused_test_probs = best_w * test_spat + (1.0 - best_w) * test_spec
fused_test_preds = (fused_test_probs < 0.50).astype(int)

final_test_acc = accuracy_score(test_true, fused_test_preds)
final_test_auc = roc_auc_score(test_true == 0, fused_test_probs)

print(f'FaceForensics++ Fused Test Accuracy: {final_test_acc*100:.2f}%')
print(f'FaceForensics++ Fused Test ROC-AUC:  {final_test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(test_true, fused_test_preds, target_names=['DEEPFAKE', 'REAL']))

# Breakdown by Manipulation Technique
print('\nAccuracy Breakdown by Manipulation Category on Test Set:')
for cat_name in ['deepfakes', 'face2face', 'faceswap', 'neuraltextures', 'original']:
    idx = [i for i, p in enumerate(test_paths) if cat_name in p.lower()]
    if idx:
        cat_acc = accuracy_score(test_true[idx], fused_test_preds[idx])
        print(f'  - {cat_name.upper():<16} ({len(idx)} test frames): Accuracy = {cat_acc*100:.2f}%')

# Export Checkpoint
export_path = SAVE_DIR / 'verisight_faceforensics_clean_production.pth'
torch.save({
    'spatial_state_dict': model_spatial.state_dict(),
    'spectral_state_dict': model_spectral.state_dict(),
    'w_spatial': best_w,
    'w_spectral': 1.0 - best_w,
    'val_accuracy': best_val_acc,
    'test_accuracy': final_test_acc,
    'test_auc': final_test_auc,
    'dataset': 'FaceForensics++ (Video-Disjoint, Frame Deduplicated)',
    'architecture': 'ResNet50_Spatial + DedicatedSpectralCNN',
    'class_to_idx': {'DEEPFAKE': 0, 'REAL': 1}
}, str(export_path))
print(f'\nSUCCESS: FaceForensics++ clean model exported to: {export_path}')